<a href="https://colab.research.google.com/github/niveditha-bh/WorkflowLLM/blob/main/dataset_reduced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers peft bitsandbytes accelerate datasets

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import json

BASE = '/content/drive/MyDrive/dataset'

with open(f'{BASE}/dataset_split_keys.json') as f:
    split_keys = json.load(f)

with open(f'{BASE}/synthesized_data.json') as f:
    synth = json.load(f)

with open(f'{BASE}/seed_data.json') as f:
    seed = json.load(f)

# Inspect structure
print("split_keys type:", type(split_keys))
print("split_keys preview:", split_keys if isinstance(split_keys, list) else list(split_keys.keys()) if isinstance(split_keys, dict) else "unknown")
print()
print("synth count:", len(synth))
print("synth first entry:", synth[0])
print()
print("seed count:", len(seed))
print("seed first entry:", seed[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
split_keys type: <class 'dict'>
split_keys preview: ['train', 'reward', 'dev', 'test']

synth count: 95810
synth first entry: {'query': 'I want to develop a feature in my photo editing app that allows users to select a specific photo from their library, apply a filter to enhance its appearance, and then share it directly to their social media accounts. The process should include the following steps: first, the user should be prompted to choose a photo from their library; next, they should be able to apply a filter of their choice; after that, the app should ask for confirmation before sharing the edited photo; finally, if the user confirms, the app should automatically post the photo to their selected social media platform. How can I implement this workflow effectively?', 'apis': ['is_workflow_actions_filter_photos', 'is_workflow_actions_choosefromlist', 'is_

In [3]:
# Check the train split size and a few examples
print("Train split size:", len(split_keys['train']))
print("Sample train keys:", split_keys['train'][:5])

# Check how many seed_data entries have a 'key' matching the train split
seed_keys = set(item['key'] for item in seed)
train_key_set = set(split_keys['train'])
overlap = seed_keys & train_key_set
print("Seed keys matching train split:", len(overlap), "out of", len(seed))

# Check whether synth data keys ever appear in the split file at all
synth_keys = set(item['key'] for item in synth)
print("Unique synth key values:", synth_keys if len(synth_keys) < 5 else f"{len(synth_keys)} unique values")

Train split size: 9757
Sample train keys: ['Advanced Clicker', 'Air Defense Drill (Stand-alone Version)', 'Alexia', 'AMMK ( by KeDa )', 'App Store tools']
Seed keys matching train split: 9757 out of 14523
Unique synth key values: 2382 unique values


In [4]:
# Check overlap of synth keys against ALL splits, not just train
all_split_keys = set()
for split_name in split_keys:
    all_split_keys.update(split_keys[split_name])

synth_overlap = synth_keys & all_split_keys
print("Synth keys matching ANY split entry:", len(synth_overlap), "out of", len(synth_keys), "unique synth keys")

# Also check synth against train specifically
synth_train_overlap = synth_keys & train_key_set
print("Synth keys matching TRAIN split specifically:", len(synth_train_overlap))

# And check dev/test just to be thorough
print("Dev split size:", len(split_keys['dev']))
print("Test split size:", len(split_keys['test']))
print("Reward split size:", len(split_keys['reward']))

Synth keys matching ANY split entry: 2380 out of 2382 unique synth keys
Synth keys matching TRAIN split specifically: 0
Dev split size: 1190
Test split size: 1190
Reward split size: 2380


In [5]:
# Filter seed_data to only the train split
seed_train = [item for item in seed if item['key'] in train_key_set]
print("Seed train examples:", len(seed_train))

# Combine with ALL of synthesized_data
full_train_data = seed_train + synth
print("Total combined training examples:", len(full_train_data))

# Now take your 5% random sample
import random
random.seed(42)
random.shuffle(full_train_data)

subset_size = int(len(full_train_data) * 0.05)
subset = full_train_data[:subset_size]
print(f"5% subset size: {len(subset)}")

# Save it
with open('/content/drive/MyDrive/dataset/train_5pct.json', 'w') as f:
    json.dump(subset, f, indent=2)

print("Saved to Drive.")

Seed train examples: 9757
Total combined training examples: 105567
5% subset size: 5278
Saved to Drive.


In [6]:
def format_example(ex):
    query = ex.get("query", "")
    workflow_code = ex.get("workflow_code", "")
    task_plan = ex.get("task_plan", "")

    if task_plan:
        text = f"### Task:\n{query}\n\n### Plan:\n{task_plan}\n\n### Workflow Code:\n{workflow_code}"
    else:
        text = f"### Task:\n{query}\n\n### Workflow Code:\n{workflow_code}"

    return text

# Load your 5% subset
import json
with open('/content/drive/MyDrive/dataset/train_5pct.json') as f:
    subset = json.load(f)

texts = [format_example(ex) for ex in subset]

# Sanity check
print(texts[0])
print("---")
print(f"Total formatted examples: {len(texts)}")
print(f"Example lengths (characters): min={min(len(t) for t in texts)}, max={max(len(t) for t in texts)}, avg={sum(len(t) for t in texts)//len(texts)}")

### Task:
How can I create a health and fitness tracking system that allows users to log their daily workouts, including the type of exercise, duration, and calories burned? This system should also enable users to set reminders for their workouts, search for nearby gyms or fitness classes, and provide a summary of their weekly progress, including total workouts and calories burned. Additionally, I want to incorporate a feature that allows users to input their fitness goals and receive motivational notifications to keep them on track.

### Plan:
1. **Start**: The process begins with a user prompt asking their intended action.
2. **User Decision Process**: Based on user input, execute corresponding actions for workout logging, reminders, gym search, weekly summaries, fitness goals or exiting.
   - **Workout Logging**: Capture workout type, duration, calories, and location with HomeKit actions.
   - **Setting Reminders**: Gather reminder time, workout type, location, and intended calories

In [7]:
import numpy as np

lengths = [len(t) for t in texts]
percentiles = [50, 75, 90, 95, 99]
for p in percentiles:
    print(f"{p}th percentile: {int(np.percentile(lengths, p))} characters")

# How many examples exceed a few candidate cutoffs?
for cutoff in [2000, 4000, 8000, 12000]:
    count = sum(1 for l in lengths if l > cutoff)
    print(f"Examples over {cutoff} chars: {count} ({count/len(lengths)*100:.1f}%)")

50th percentile: 5118 characters
75th percentile: 6907 characters
90th percentile: 9775 characters
95th percentile: 13359 characters
99th percentile: 38799 characters
Examples over 2000 chars: 5193 (98.4%)
Examples over 4000 chars: 3959 (75.0%)
Examples over 8000 chars: 896 (17.0%)
Examples over 12000 chars: 334 (6.3%)


In [8]:
from huggingface_hub import login
login()  # paste the token from YOUR approved Hugging Face account here

In [9]:
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

from datasets import Dataset

dataset = Dataset.from_dict({"text": texts})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=2048,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized_dataset)
print("Sample token count:", len(tokenized_dataset[0]["input_ids"]))

Map:   0%|          | 0/5278 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5278
})
Sample token count: 2048


In [10]:
from huggingface_hub import login
login()  # paste the token from YOUR approved Hugging Face account here

In [11]:
!pip install -U "bitsandbytes>=0.46.1"

In [12]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "meta-llama/Llama-3.1-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # changed from bfloat16
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully.")
print("Memory footprint:", model.get_memory_footprint() / 1e9, "GB")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully.
Memory footprint: 5.591540224 GB


In [13]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [15]:
training_args = TrainingArguments(
    output_dir="/content/workflowllm-lora-5pct",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,          # reduced from 3 to 1
    max_steps=100,               # NEW: hard cap — stop after 100 steps regardless
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.116068
20,1.044263
30,0.999671
40,0.997976
50,1.020777
60,0.914400


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.116068
20,1.044263
30,0.999671
40,0.997976
50,1.020777
60,0.914400
70,0.941268
80,0.955249
90,0.929246
100,0.904994


TrainOutput(global_step=100, training_loss=0.9823913097381591, metrics={'train_runtime': 7024.8727, 'train_samples_per_second': 0.114, 'train_steps_per_second': 0.014, 'total_flos': 7.38434129854464e+16, 'train_loss': 0.9823913097381591, 'epoch': 0.15157256536566882})